# Impute some field
**Warning**: if you really hate yourself then **run this file and wait for about 6 hours** or you can just use the cleaned version of the datafile

In [ ]:
import time
import requests
import json
import re
import pandas as pd
from llama_index.llms.ollama import Ollama
from llama_index.core.llms import ChatMessage
from utils.preprocess.imputation.crawler import fetch_steam_app_data

In [ ]:
DATA_PATH = 'data/steam-games/SteamGames.csv'
df = pd.read_csv(DATA_PATH)

### Impuate Description

In [ ]:
target_cols = ["Description", "Genres", "OsRequirement", "MemoryRequirement", "CpuRequirement"]
missing_mask = df[target_cols].isna().any(axis=1)
missing_indices = df[missing_mask].index


print(f"Found {len(missing_indices)} rows needing API enrichment.")

session = requests.Session()
print(f'Request session complete')
for count, idx in enumerate(missing_indices, start=1):
    appId = int(df.loc[idx, 'Appid'])
    crawled = fetch_steam_app_data(appId, session)
    print(f'Fetch {count} complete')

    if crawled:
        for col in target_cols:
            if pd.isna(df.loc[idx, col]) and crawled.get(col) is not None:
                df.loc[idx, col] = crawled[col]

    time.sleep(3.0)

    if count % 100 == 0:
        # checkpoint
        df.to_csv('data/steam_checkpoint.csv', index=False)
        print(f"Checkpointed {count}/{len(missing_indices)} rows.")
print(f"Complete imputate!")

Found 3770 rows needing API enrichment.
Request session complete
Fetch 1 complete
Fetch 2 complete
Fetch 3 complete
Fetch 4 complete
Fetch 5 complete
Fetch 6 complete
Fetch 7 complete
Fetch 8 complete
Fetch 9 complete
Fetch 10 complete
Fetch 11 complete
Fetch 12 complete
Fetch 13 complete
Fetch 14 complete
Fetch 15 complete
Fetch 16 complete
Fetch 17 complete
Fetch 18 complete
Fetch 19 complete
Fetch 20 complete
Fetch 21 complete
Fetch 22 complete
Fetch 23 complete
Fetch 24 complete
Fetch 25 complete
Fetch 26 complete
Fetch 27 complete
Fetch 28 complete
Fetch 29 complete
Fetch 30 complete
Fetch 31 complete
Fetch 32 complete
Fetch 33 complete
Fetch 34 complete
Fetch 35 complete
Fetch 36 complete
Fetch 37 complete
Fetch 38 complete
Fetch 39 complete
Fetch 40 complete
Fetch 41 complete
Fetch 42 complete
Fetch 43 complete
Fetch 44 complete
Fetch 45 complete
Fetch 46 complete
Fetch 47 complete
Fetch 48 complete
Fetch 49 complete
Fetch 50 complete
Fetch 51 complete
Fetch 52 complete
Fetch 53

### Fallback fill out any remain using LLM

In [ ]:
# 1. Initialize Ollama LLM
llm = Ollama(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    request_timeout=600.0,
    json_mode=True,
    temperature=0.2,  # Low temperature keeps descriptions factual and grounded
)

# 2. Complete System & User Prompts (Notice doubled {{ }} for literal JSON braces)
SYSTEM_PROMPT = """
You are a video game catalog archivist. Your task is to write a concise, factual, 2-to-4 sentence Steam store description based strictly on the provided game metadata.
Rules:
- Focus on the game's core genre, themes, and gameplay style implied by the metadata.
- Do NOT invent specific story character names, fake release dates, or unverified features.
- Output ONLY a valid JSON object with a single key "description".
""".strip()

USER_PROMPT = """
Write a short and sweet Steam store description for "{game_name}" (maximum 150 words).
Name: {game_name}
Type: {game_type}
Genres: {game_genres}
Tags: {game_tags}
Developer: {game_developer}
Publisher: {game_publisher}

Strictly follow the JSON output format below:
{{
    "description": "Your concise game description here."
}}
""".strip()

In [ ]:
keep_features = ["game", "dlc"]
df = df[df['Type'].isin(keep_features)].reset_index(drop=True)

# Check for any rows in Description feature missing
missing_mask = df['Description'].isna()
missing_indices = df[missing_mask].index

In [ ]:
def safe_val(val, default="Unknown") -> str:
    """Helper to convert NaN/None cells to a clean string before formatting."""
    return str(val).strip() if pd.notna(val) and str(val).strip() else default

# 3. Enrich missing rows
missing_indices = df[df["Description"].isna()].index
print(f"Found {len(missing_indices)} rows needing LLM generation enrichment.")

for count, index in enumerate(missing_indices, start=1):
    row = df.loc[index]
    
    # Fill out the user prompt using safe_val to handle any NaN columns
    formatted_user_prompt = USER_PROMPT.format(
        game_name=safe_val(row.get("Name")),
        game_type=safe_val(row.get("Type"), default="game"),
        game_genres=safe_val(row.get("Genres")),
        game_tags=safe_val(row.get("Tags")),
        game_developer=safe_val(row.get("Developers")),
        game_publisher=safe_val(row.get("Publishers")),
    )

    messages = [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=formatted_user_prompt),
    ]

    try:
        # Use .chat() for ChatMessage lists
        response = llm.chat(messages)
        raw_content = response.message.content.strip()
        
        # Strip <think>...</think> blocks just in case the model outputs reasoning tags
        clean_json_str = re.sub(r"<think>.*?</think>", "", raw_content, flags=re.DOTALL).strip()
        
        # Parse JSON string into a Python dict
        data = json.loads(clean_json_str)
        description_text = data.get("description", "").strip()
        
        if description_text:
            df.loc[index, "Description"] = description_text
            print(f"[{count}/{len(missing_indices)}] Generated description for: {row.get('Name')}")
        else:
            print(f"[{count}/{len(missing_indices)}] Warning: Empty description returned for index {index}")

    except Exception as e:
        print(f"[{count}/{len(missing_indices)}] Error on index {index} ({row.get('Name')}): {e}")

    # Save checkpoint every 10 rows (and on the final row)
    if count % 10 == 0 or count == len(missing_indices):
        df.to_csv("data/fallback_check_point.csv", index=False)
        print(f"Checkpointed {count}/{len(missing_indices)} rows.")

df.to_csv("data/steam-games/SteamGames_Clean_1.csv", index=False)
print("Generation impute complete.")

Found 2 rows needing LLM generation enrichment.
[1/2] Generated description for: The Evil Within: The Executioner
[2/2] Generated description for: TOXIKK - [UPGRADE] - Free Edition to FULL GAME
Checkpointed 2/2 rows.
Generation impute complete.
